In [1]:
import gc

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns



load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
prev_app_df = pd.read_parquet(cfg.PROCESSED_DIR / "previous_application.train-processed.parquet")


merged_df = application_train_df.merge(
    prev_app_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train + prev_application")
#auc_score_OOF= 0.754

#freeing memory
del merged_df
gc.collect()



[0]	validation_0-auc:0.71367
[1]	validation_0-auc:0.72475
[2]	validation_0-auc:0.72876
[3]	validation_0-auc:0.73188
[4]	validation_0-auc:0.73575
[5]	validation_0-auc:0.73836
[6]	validation_0-auc:0.74116
[7]	validation_0-auc:0.74387
[8]	validation_0-auc:0.74634
[9]	validation_0-auc:0.74739
[10]	validation_0-auc:0.74941
[11]	validation_0-auc:0.75088
[12]	validation_0-auc:0.75159
[13]	validation_0-auc:0.75233
[14]	validation_0-auc:0.75353
[15]	validation_0-auc:0.75431
[16]	validation_0-auc:0.75581
[17]	validation_0-auc:0.75612
[18]	validation_0-auc:0.75685
[19]	validation_0-auc:0.75725
[20]	validation_0-auc:0.75754
[21]	validation_0-auc:0.75786
[22]	validation_0-auc:0.75828
[23]	validation_0-auc:0.75863
[24]	validation_0-auc:0.75856
[25]	validation_0-auc:0.75874
[26]	validation_0-auc:0.75878
[27]	validation_0-auc:0.75855
[28]	validation_0-auc:0.75892
[29]	validation_0-auc:0.75900
[30]	validation_0-auc:0.75892
[31]	validation_0-auc:0.75897
[32]	validation_0-auc:0.75927
[33]	validation_0-au

442

In [4]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train + prev_application + installment")
#auc_score_OOF= 0.76

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.71487
[1]	validation_0-auc:0.72599
[2]	validation_0-auc:0.73130
[3]	validation_0-auc:0.73421
[4]	validation_0-auc:0.73724
[5]	validation_0-auc:0.74115
[6]	validation_0-auc:0.74320
[7]	validation_0-auc:0.74600
[8]	validation_0-auc:0.74995
[9]	validation_0-auc:0.75216
[10]	validation_0-auc:0.75444
[11]	validation_0-auc:0.75627
[12]	validation_0-auc:0.75776
[13]	validation_0-auc:0.75890
[14]	validation_0-auc:0.75993
[15]	validation_0-auc:0.76044
[16]	validation_0-auc:0.76173
[17]	validation_0-auc:0.76259
[18]	validation_0-auc:0.76287
[19]	validation_0-auc:0.76339
[20]	validation_0-auc:0.76375
[21]	validation_0-auc:0.76392
[22]	validation_0-auc:0.76422
[23]	validation_0-auc:0.76432
[24]	validation_0-auc:0.76408
[25]	validation_0-auc:0.76416
[26]	validation_0-auc:0.76391
[27]	validation_0-auc:0.76448
[28]	validation_0-auc:0.76439
[29]	validation_0-auc:0.76453
[30]	validation_0-auc:0.76461
[31]	validation_0-auc:0.76467
[32]	validation_0-auc:0.76442
[33]	validation_0-au

359

In [ ]:
#for the second one  we gonna analize the gains from the aggregation of bureau
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train+bureau")
#auc_score_OOF= 0.752

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.71408
[1]	validation_0-auc:0.72260
[2]	validation_0-auc:0.72744
[3]	validation_0-auc:0.72960
[4]	validation_0-auc:0.73170
[5]	validation_0-auc:0.73427
[6]	validation_0-auc:0.73735
[7]	validation_0-auc:0.73969
[8]	validation_0-auc:0.74285
[9]	validation_0-auc:0.74424
[10]	validation_0-auc:0.74548
[11]	validation_0-auc:0.74787
[12]	validation_0-auc:0.74857
[13]	validation_0-auc:0.74952
[14]	validation_0-auc:0.75146
[15]	validation_0-auc:0.75238
[16]	validation_0-auc:0.75305
[17]	validation_0-auc:0.75345
[18]	validation_0-auc:0.75397
[19]	validation_0-auc:0.75441
[20]	validation_0-auc:0.75459
[21]	validation_0-auc:0.75521
[22]	validation_0-auc:0.75525
[23]	validation_0-auc:0.75543
[24]	validation_0-auc:0.75561
[25]	validation_0-auc:0.75540
[26]	validation_0-auc:0.75569
[27]	validation_0-auc:0.75624
[28]	validation_0-auc:0.75641
[29]	validation_0-auc:0.75631
[30]	validation_0-auc:0.75622
[31]	validation_0-auc:0.75615
[32]	validation_0-auc:0.75588
[33]	validation_0-au

691

In [5]:
#finally we try with app_train + prev_app + bureau
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments.parquet")


merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.0 (app_train+bureau+prev_app+installments)")
#auc_score_OOF= 0.760 is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.71491
[1]	validation_0-auc:0.72610
[2]	validation_0-auc:0.73168
[3]	validation_0-auc:0.73467
[4]	validation_0-auc:0.73792
[5]	validation_0-auc:0.74137
[6]	validation_0-auc:0.74431
[7]	validation_0-auc:0.74740
[8]	validation_0-auc:0.75002
[9]	validation_0-auc:0.75237
[10]	validation_0-auc:0.75513
[11]	validation_0-auc:0.75736
[12]	validation_0-auc:0.75977
[13]	validation_0-auc:0.76076
[14]	validation_0-auc:0.76158
[15]	validation_0-auc:0.76298
[16]	validation_0-auc:0.76319
[17]	validation_0-auc:0.76403
[18]	validation_0-auc:0.76466
[19]	validation_0-auc:0.76463
[20]	validation_0-auc:0.76499
[21]	validation_0-auc:0.76560
[22]	validation_0-auc:0.76649
[23]	validation_0-auc:0.76640
[24]	validation_0-auc:0.76716
[25]	validation_0-auc:0.76768
[26]	validation_0-auc:0.76769
[27]	validation_0-auc:0.76774
[28]	validation_0-auc:0.76791
[29]	validation_0-auc:0.76787
[30]	validation_0-auc:0.76802
[31]	validation_0-auc:0.76812
[32]	validation_0-auc:0.76827
[33]	validation_0-au

400

In [3]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_feature_engineering.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments.parquet")


merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.1 (app_train_with_features+bureau+prev_app+installments)")
#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.71237
[1]	validation_0-auc:0.73080
[2]	validation_0-auc:0.73482
[3]	validation_0-auc:0.73806
[4]	validation_0-auc:0.74147
[5]	validation_0-auc:0.74492
[6]	validation_0-auc:0.74821
[7]	validation_0-auc:0.75112
[8]	validation_0-auc:0.75263
[9]	validation_0-auc:0.75550
[10]	validation_0-auc:0.75636
[11]	validation_0-auc:0.75912
[12]	validation_0-auc:0.76082
[13]	validation_0-auc:0.76176
[14]	validation_0-auc:0.76299
[15]	validation_0-auc:0.76369
[16]	validation_0-auc:0.76510
[17]	validation_0-auc:0.76553
[18]	validation_0-auc:0.76583
[19]	validation_0-auc:0.76621
[20]	validation_0-auc:0.76697
[21]	validation_0-auc:0.76768
[22]	validation_0-auc:0.76771
[23]	validation_0-auc:0.76820
[24]	validation_0-auc:0.76853
[25]	validation_0-auc:0.76866
[26]	validation_0-auc:0.76912
[27]	validation_0-auc:0.76966
[28]	validation_0-auc:0.76975
[29]	validation_0-auc:0.77012
[30]	validation_0-auc:0.77007
[31]	validation_0-auc:0.77019
[32]	validation_0-auc:0.77069
[33]	validation_0-au

449